> 在线讲义：[第 30 章 · DeepSeekMoE：专家切细、共享专家，以及不加 loss 的负载均衡](https://wikieden.github.io/beyond-transformer/lessons/30-deepseekmoe.html) · 本书《超越 Transformer》

# 第 30 章 · DeepSeekMoE：专家切细、共享专家，以及不加 loss 的负载均衡

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/30-deepseekmoe.html) · 仓库里 `beyond-transformer/notes/lessons/30-deepseekmoe.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 组合数 | C(mN, mK) | `fine_grained()` | 120 → 4,426,165,368 |
| 第三步 bias 均衡 | Algorithm 1 | `simulate(use_bias=...)` | 最大 / 平均 2.36 → 1.03 |
| 第三步 门控权重 | 原始分数 | `gate_value(s, [0, 1])` | [0.667, 0.333, 0, 0] |

不加 loss、只调 bias 的负载均衡：b_i ← b_i + u·sign(平均负载 − 专家 i 的负载)，bias 只影响“选谁”，不进权重和梯度。
在自己的 MoE 上比较三种均衡方法、画负载直方图是里程碑 T-M6。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
from math import comb

**函数 `combos`**：小工具。

In [ ]:
def combos(n, k):
    return comb(n, k)

**函数 `fine_grained`**：每个专家切成 m 份（中间层 d_ff/m），激活 mK 份：参数和计算量不变，组合数从 C(N, K) 变成 C(mN, mK)。

In [ ]:
def fine_grained(N=16, K=2, m=4, d_ff=1024, d=256):
    """每个专家切成 m 份（中间层 d_ff/m），激活 mK 份：参数和计算量不变，组合数从 C(N, K) 变成 C(mN, mK)。"""
    params_before = N * 3 * d * d_ff
    params_after = m * N * 3 * d * (d_ff // m)
    return combos(N, K), combos(m * N, m * K), params_before, params_after

**函数 `simulate`**：router 分数 = 每个专家固定的偏好 + 噪声；偏好让前几个专家天然更受欢迎。每步按 top-k 选专家、统计负载、更新 bias。

In [ ]:
def simulate(steps=300, T=512, E=8, k=2, u=0.01, seed=0, use_bias=True):
    """router 分数 = 每个专家固定的偏好 + 噪声；偏好让前几个专家天然更受欢迎。每步按 top-k 选专家、统计负载、更新 bias。
    返回：第一步和最后一步的负载（占比）、最大负载 / 平均负载（MaxVio + 1）。"""
    rng = np.random.default_rng(seed)
    pref = np.linspace(1.0, -1.0, E)
    b = np.zeros(E)
    hist = []
    for _ in range(steps):
        s = pref + rng.normal(0, 1.0, (T, E))
        chosen = np.argsort(-(s + b), axis=1)[:, :k]          # bias 只用来选
        load = np.bincount(chosen.ravel(), minlength=E) / (T * k)
        hist.append(load)
        if use_bias:
            b += u * np.sign(load.mean() - load)               # 负载低于平均 → bias 加大
    first, last = hist[0], np.mean(hist[-20:], 0)
    return first, last, float(last.max() / last.mean()), b

**函数 `gate_value`**：门控权重仍然用原始分数 s（不带 bias）：bias 不改变“加权多少”，只改变“选谁”。

In [ ]:
def gate_value(s, chosen):
    """门控权重仍然用原始分数 s（不带 bias）：bias 不改变“加权多少”，只改变“选谁”。"""
    g = np.zeros_like(s)
    g[chosen] = s[chosen] / s[chosen].sum()
    return g

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 细粒度切分：组合数")
c1, c2, p1, p2 = fine_grained()
print(f"  16 个专家选 2：{c1:,} 种组合；每个切成 4 份、64 个选 8：{c2:,} 种；参数 {p1:,} → {p2:,}（不变）")
print("\n第二步 共享专家")
print("  DeepSeek-V2：每层 2 个共享专家 + 160 个路由专家选 6；共享专家放所有 token 都需要的通用知识，路由专家各自专精")

In [ ]:
print("\n第三步 不加 loss、只调 bias（8 个专家选 2，u = 0.01）")
f0, l0, v0, _ = simulate(use_bias=False)
f1, l1, v1, b = simulate(use_bias=True)
print(f"  不调 bias：负载 {np.round(l0, 3).tolist()}，最大 / 平均 = {v0:.2f}")
print(f"  调 bias 300 步：负载 {np.round(l1, 3).tolist()}，最大 / 平均 = {v1:.2f}")
print(f"  学到的 bias：{np.round(b, 2).tolist()}（天然受欢迎的专家被减分）")
s = np.array([0.6, 0.3, 0.1, 0.05])
print(f"  门控权重只用原始分数：分数 {s.tolist()}、选 [0, 1] → {np.round(gate_value(s, [0, 1]), 3).tolist()}")

## 3. 代码里才看得到的

- **router 分数是“固定偏好 + 噪声”的玩具**：真实 router 的分数在训练中一直在变，bias 要一边追一边调；这里的偏好不变，所以 bias 会收敛。
- **sign 让更新步长固定**：不管偏差多大，每步只挪 u；所以 u 太小追不上、太大会来回抖。调 u 是这个方法唯一的超参。

## 4. 自己改一改

- 把 `simulate` 的 u 改成 0.1，最大 / 平均和 bias 怎么变？负载会抖吗？
- 把偏好 `pref` 从 linspace(1, −1) 改成 linspace(3, −3)，300 步够不够推平？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 大积木拆小积木，选得更多，组合从一百二十涨到四十四亿；参数和计算都不变。
> 公共知识交给共享专家，人人都走；路由专家各自专精。
> 不加 loss 只调 bias：负载低就加、高就减，只管选谁、不管加权，不进梯度，均衡不扰主任务。